<a href="https://colab.research.google.com/github/Cloud-Architect-Emma/3-Tier-Web-App-on-AWS/blob/main/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import sys
import torch
import platform

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("Operating system:", platform.system())
print("GPU available:", torch.cuda.is_available())

print("\nMiniLLM Research Lab is ready!")


Python version: 3.13.18
PyTorch version: 2.14.1+cpu
Operating system: Linux
GPU available: False

MiniLLM Research Lab is ready!


In [2]:

from pathlib import Path
from urllib.request import urlretrieve

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "tiny_shakespeare.txt"

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "karpathy/char-rnn/master/data/"
    "tinyshakespeare/input.txt"
)

if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

text = DATA_PATH.read_text(encoding="utf-8")

print("Download complete!")
print("Total characters:", len(text))
print("First 500 characters:")
print(text[:500])


Download complete!
Total characters: 1115394
First 500 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:

print("Dataset length:", len(text))

print(
    "Number of lines:",
    len(text.splitlines())
)

unique_characters = sorted(set(text))

print(
    "Unique characters:",
    len(unique_characters)
)

print(
    "Character vocabulary:",
    repr("".join(unique_characters))
)

print("\nFirst 300 characters:")
print(text[:300])


Dataset length: 1115394
Number of lines: 40000
Unique characters: 65
Character vocabulary: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

First 300 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [4]:

chars = sorted(set(text))

vocab_size = len(chars)

stoi = {
    character: index
    for index, character in enumerate(chars)
}

itos = {
    index: character
    for character, index in stoi.items()
}

def encode(message):
    return [stoi[c] for c in message]

def decode(token_ids):
    return "".join(itos[i] for i in token_ids)

sample = "Hello"

encoded = encode(sample)
decoded = decode(encoded)

print("Original:", sample)
print("Encoded:", encoded)
print("Decoded:", decoded)
print("Vocabulary size:", vocab_size)


Original: Hello
Encoded: [20, 43, 50, 50, 53]
Decoded: Hello
Vocabulary size: 65


In [5]:

test_messages = [
    "Hello",
    "The king",
    "To be, or not to be"
]

for message in test_messages:
    recovered = decode(encode(message))

    assert recovered == message

    print(
        f"PASS: {message!r}"
    )

print("\nAll tokenizer tests passed!")


PASS: 'Hello'
PASS: 'The king'
PASS: 'To be, or not to be'

All tokenizer tests passed!


In [6]:

import torch

all_tokens = torch.tensor(
    encode(text),
    dtype=torch.long
)

split_index = int(0.9 * len(all_tokens))

train_data = all_tokens[:split_index]
val_data = all_tokens[split_index:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))
print("Vocabulary size:", vocab_size)

assert len(train_data) > 0
assert len(val_data) > 0
assert len(train_data) + len(val_data) == len(all_tokens)

print("Dataset split validated!")


Training tokens: 1003854
Validation tokens: 111540
Vocabulary size: 65
Dataset split validated!


In [7]:

block_size = 8

def get_example(data, start_index=0):
    x = data[
        start_index:start_index + block_size
    ]

    y = data[
        start_index + 1:
        start_index + block_size + 1
    ]

    return x, y

x, y = get_example(train_data)

print("Input token IDs:", x.tolist())
print("Target token IDs:", y.tolist())

print("\nInput text:", repr(decode(x.tolist())))
print("Target text:", repr(decode(y.tolist())))

assert len(x) == block_size
assert len(y) == block_size
assert torch.equal(x[1:], y[:-1])

print("\nNext-token alignment validated!")


Input token IDs: [18, 47, 56, 57, 58, 1, 15, 47]
Target token IDs: [47, 56, 57, 58, 1, 15, 47, 58]

Input text: 'First Ci'
Target text: 'irst Cit'

Next-token alignment validated!


In [8]:

import json

torch.save(train_data, DATA_DIR / "train.pt")
torch.save(val_data, DATA_DIR / "val.pt")

tokenizer_config = {
    "characters": chars,
    "vocab_size": vocab_size,
    "tokenizer_type": "character"
}

with open(
    DATA_DIR / "tokenizer.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        tokenizer_config,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Prepared datasets saved!")
print("Tokenizer configuration saved!")

print("\nFiles:")
for file in DATA_DIR.iterdir():
    print("-", file.name)


Prepared datasets saved!
Tokenizer configuration saved!

Files:
- tiny_shakespeare.txt
- val.pt
- train.pt
- tokenizer.json


In [9]:

assert vocab_size == len(stoi)
assert vocab_size == len(itos)

assert decode(encode("Hello")) == "Hello"

assert len(train_data) > len(val_data)

assert int(train_data.min()) >= 0
assert int(train_data.max()) < vocab_size

assert int(val_data.min()) >= 0
assert int(val_data.max()) < vocab_size

assert (DATA_DIR / "train.pt").exists()
assert (DATA_DIR / "val.pt").exists()
assert (DATA_DIR / "tokenizer.json").exists()

print("ALL LAB 1 VALIDATION TESTS PASSED")


ALL LAB 1 VALIDATION TESTS PASSED
